# Hybrid UDE for forecasting tumor response under immunotherapy

Code for the IJCM manuscript *Amortized Hybrid Neural Differential Equations with Symbolic Recovery for Forecasting Tumor Response to Immunotherapy* (A. Nargis, DHA Suffa University).

Data are the lesion measurements from Ghaffari Laleh et al. (2022, *PLOS Comput. Biol.*): five atezolizumab / docetaxel trials in NSCLC and urothelial cancer. I use the copy that ships with TumorGrowth.jl (MIT licence), so nothing needs to be downloaded by hand.

Written for the free Colab CPU runtime. `Runtime > Run all` takes roughly 45-50 min with the default settings; set `FAST = True` below for a ~5 min test run. Tables, figures, models, predictions and a `summary.json` go to a time-stamped folder in Drive.

## Setup

In [ ]:
import os, sys, json, time, math, warnings, platform, hashlib, io, urllib.request, random, copy

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import scipy
from scipy.optimize import least_squares
from scipy.stats import wilcoxon
import sklearn
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.exceptions import ConvergenceWarning
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

try:
    from IPython.display import display, Markdown
except Exception:  # not in jupyter
    display = print
    Markdown = str

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ---- settings ----
FAST = False  # True = quick test run (1 seed, few epochs, ~5 min)

CFG = dict(
    SEED=0,
    N_SEEDS=3,                # seeds for every neural model
    TIME_UNIT_DAYS=100.0,     # 1 model time unit = 100 days
    VOL_OFFSET=65.0,          # mm^3 (5 mm lesion), keeps the log finite if a lesion disappears
    DT=0.10,                  # RK4 step = 10 days
    MIN_OBS_EVAL=6,           # need >= 6 scans to be in cal/test
    N_HOLDOUT_P1=3,
    N_EARLY_P2=3,
    SPLIT_TRAIN_CAL_TEST=(0.40, 0.25, 0.35),
    HIDDEN=32,
    AMORT_EPOCHS=400,
    AMORT_LR=3e-3,
    BATCH=256,
    SYMB_EPOCHS=150,          # fine-tuning after symbolic recovery
    EPOCHS_POP=600,           # population fit for UDE-MAP / Stein-EB
    EPOCHS_ADAPT=250,         # per-lesion MAP steps
    LR_NET=5e-3,
    LR_PAT=3e-2,
    LR_ADAPT=5e-2,
    ALPHA=0.10,               # 90% bands
    H0=0.10,
    NI_MARGIN=0.05,           # non-inferiority margin vs LOCF (log units)
    Z_MIN=-10.0, Z_MAX=6.0,
    N_BOOT=1000,
    N_RESPLIT=500,
    STLSQ_R2_FRACTION=0.95,   # keep >= 95% of the full-library R^2
)
if FAST:
    CFG.update(N_SEEDS=1, AMORT_EPOCHS=60, SYMB_EPOCHS=30, EPOCHS_POP=120, EPOCHS_ADAPT=80, N_BOOT=200, N_RESPLIT=100)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

set_seed(CFG["SEED"])
torch.set_num_threads(max(1, os.cpu_count() or 1))
torch.set_default_dtype(torch.float32)
DEVICE = torch.device("cpu")


# ---- drive + output folders ----
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE_DIR = "/content/drive/MyDrive/IJCM_UDE_Immunotherapy"
else:
    BASE_DIR = os.path.abspath("./IJCM_UDE_Immunotherapy")

RUN_ID = time.strftime("run_%Y%m%d_%H%M%S")
OUT = os.path.join(BASE_DIR, RUN_ID)
DIRS = {k: os.path.join(OUT, k) for k in ["figures", "tables", "models", "data", "predictions", "logs"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
CACHE_DIR = os.path.join(BASE_DIR, "data_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

LOG_FILE = os.path.join(DIRS["logs"], "run_log.txt")
T_START = time.time()
TIMINGS = {}

def log(msg):
    line = f"[{time.time() - T_START:7.1f}s] {msg}"
    print(line)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

ENV = dict(python=sys.version.split()[0], platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__,
           scipy=scipy.__version__, sklearn=sklearn.__version__, torch=torch.__version__, matplotlib=matplotlib.__version__,
           cpu_threads=torch.get_num_threads(), colab=IN_COLAB)
with open(os.path.join(DIRS["logs"], "config.json"), "w") as f:
    json.dump(dict(config=CFG, fast=FAST, environment=ENV, run_id=RUN_ID), f, indent=2)
log(f"Results folder: {OUT}")
log(f"Environment: {ENV}")

# ---- plots ----
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY = "#a8a7a2"
INK, INK2 = "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": INK2, "axes.labelcolor": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6,
    "legend.frameon": False, "lines.linewidth": 2.0, "figure.facecolor": "white", "axes.facecolor": "white",
})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(DIRS["figures"], f"{name}.{ext}"), bbox_inches="tight")
    plt.show()
    plt.close(fig)
    log(f"Saved figure {name}")

def save_table(df, name, caption=None, index=False, show=True):
    df.to_csv(os.path.join(DIRS["tables"], f"{name}.csv"), index=index)
    try:
        tex = df.to_latex(index=index, float_format="%.4f", caption=caption, label=f"tab:{name}", escape=True)
        with open(os.path.join(DIRS["tables"], f"{name}.tex"), "w") as f:
            f.write(tex)
    except Exception as e:  # latex export is only a convenience
        log(f"LaTeX export skipped for {name}: {e}")
    if show:
        if caption:
            display(Markdown(f"**{caption}**"))
        display(df)
    log(f"Saved table {name}")

## Data
Tries GitHub first, then a CDN mirror. A copy is cached in Drive so later runs work without GitHub.

In [ ]:
DATA_URLS = [
    "https://raw.githubusercontent.com/ablaom/TumorGrowth.jl/dev/data/flat_patient_data.csv",
    "https://cdn.jsdelivr.net/gh/ablaom/TumorGrowth.jl@dev/data/flat_patient_data.csv",
    "https://raw.githubusercontent.com/ablaom/TumorGrowth.jl/main/data/flat_patient_data.csv",
]
REQUIRED_COLS = {"Pt_hashID", "Study_Arm", "Study_id", "Arm_id", "T_weeks", "T_days", "Lesion_diam", "Lesion_vol"}
CACHE_FILE = os.path.join(CACHE_DIR, "flat_patient_data.csv")

def load_raw():
    for url in DATA_URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                content = r.read()
            df = pd.read_csv(io.BytesIO(content))
            if REQUIRED_COLS.issubset(df.columns) and len(df) > 1000:
                with open(CACHE_FILE, "wb") as f:
                    f.write(content)
                log(f"Downloaded data from {url} ({len(content)/1e6:.2f} MB)")
                return df, url, hashlib.md5(content).hexdigest()
            log(f"Unexpected content at {url}")
        except Exception as e:
            log(f"Download failed from {url}: {e}")
    if os.path.exists(CACHE_FILE):
        content = open(CACHE_FILE, "rb").read()
        log("Using cached copy from Google Drive")
        return pd.read_csv(io.BytesIO(content)), "drive_cache", hashlib.md5(content).hexdigest()
    raise RuntimeError("Could not download the dataset from any mirror and no cached copy exists.")

raw, DATA_SOURCE, DATA_MD5 = load_raw()
raw.to_csv(os.path.join(DIRS["data"], "flat_patient_data_raw.csv"), index=False)
log(f"Raw rows: {len(raw)}, lesions: {raw.Pt_hashID.nunique()}, md5: {DATA_MD5}")
display(raw.head())

## Preprocessing
- time = days since the baseline scan / 100
- state z = log((V + c) / (V0 + c)) with c = 65 mm³ (a 5 mm sphere), so z(0) = 0 and a lesion that disappears stays finite
- same-day duplicate scans are averaged

In [ ]:
TU, C_OFF = CFG["TIME_UNIT_DAYS"], CFG["VOL_OFFSET"]
ZMIN, ZMAX = CFG["Z_MIN"], CFG["Z_MAX"]

df = raw.copy()
df["Study_id"] = df["Study_id"].astype(int)
df = (df.groupby(["Pt_hashID", "T_days"], as_index=False)
        .agg(Study_id=("Study_id", "first"), Arm_id=("Arm_id", "first"), Study_Arm=("Study_Arm", "first"),
             Lesion_diam=("Lesion_diam", "mean"), Lesion_vol=("Lesion_vol", "mean"))
        .sort_values(["Pt_hashID", "T_days"]).reset_index(drop=True))
df = df[np.isfinite(df.Lesion_vol) & np.isfinite(df.T_days)]

PATIENTS = []
for pid, g in df.groupby("Pt_hashID", sort=True):
    if len(g) < 3:
        continue
    tdays = g.T_days.values.astype(float)
    vol = g.Lesion_vol.values.astype(float)
    t = (tdays - tdays[0]) / TU
    z = np.log((vol + C_OFF) / (vol[0] + C_OFF))
    PATIENTS.append(dict(pid=pid, study=int(g.Study_id.iloc[0]), arm=int(g.Arm_id.iloc[0]), study_arm=g.Study_Arm.iloc[0],
                         t=t, z=z, diam=g.Lesion_diam.values.astype(float), vol=vol, n=len(g)))
NP = len(PATIENTS)
n_obs = np.array([p["n"] for p in PATIENTS])
studies = np.array([p["study"] for p in PATIENTS])
STUDIES = sorted(set(studies.tolist()))
log(f"Lesions with >= 3 scans: {NP}; with >= {CFG['MIN_OBS_EVAL']} scans: {(n_obs >= CFG['MIN_OBS_EVAL']).sum()}")

# split by lesion, stratified by trial. Short series (3-5 scans) only go to training.
idx_all = np.arange(NP)
idx_eval = idx_all[n_obs >= CFG["MIN_OBS_EVAL"]]
idx_short = idx_all[n_obs < CFG["MIN_OBS_EVAL"]]
ftr, fcal, fte = CFG["SPLIT_TRAIN_CAL_TEST"]
ev_tr, ev_rest = train_test_split(idx_eval, train_size=ftr, stratify=studies[idx_eval], random_state=CFG["SEED"])
ev_cal, ev_te = train_test_split(ev_rest, train_size=fcal / (fcal + fte), stratify=studies[ev_rest], random_state=CFG["SEED"])
SPLIT = dict(train=np.sort(np.concatenate([ev_tr, idx_short])), cal=np.sort(ev_cal), test=np.sort(ev_te))

# a few patients appear in two trials (same hash, different -S suffix).
# put all of their lesions in training so nothing leaks into cal/test
PKEY = np.array([p["pid"].rsplit("-", 1)[0] for p in PATIENTS])
_kc = pd.Series(PKEY).value_counts()
MULTI = np.isin(PKEY, _kc[_kc > 1].index.values)
_moved = np.array([i for i in np.concatenate([SPLIT["cal"], SPLIT["test"]]) if MULTI[i]], dtype=int)
SPLIT = dict(train=np.sort(np.concatenate([SPLIT["train"], _moved]).astype(int)),
             cal=np.array([i for i in SPLIT["cal"] if not MULTI[i]], dtype=int),
             test=np.array([i for i in SPLIT["test"] if not MULTI[i]], dtype=int))
for _k in PKEY[MULTI]:
    assert len(set(SPLIT_NAME for SPLIT_NAME, v in SPLIT.items() for i in v if PKEY[i] == _k)) == 1
log(f"Patients enrolled in two trials: {int((_kc > 1).sum())} ({int(MULTI.sum())} lesions); {len(_moved)} moved from calibration/test to training")
split_of = np.empty(NP, dtype=object)
for k, v in SPLIT.items():
    split_of[v] = k
log(f"Split sizes: train={len(SPLIT['train'])}, calibration={len(SPLIT['cal'])}, test={len(SPLIT['test'])}")

# each item = (lesion index, observed idx, target idx)
def make_protocol(idx, name):
    items = []
    for i in idx:
        n = PATIENTS[i]["n"]
        if name == "P1":
            k = n - CFG["N_HOLDOUT_P1"]
        else:
            k = CFG["N_EARLY_P2"]
        items.append((int(i), np.arange(k), np.arange(k, n)))
    return items

PROTOCOLS = ["P1", "P2"]
PROTO_DESC = {"P1": "P1: forecast last 3 scans", "P2": "P2: early forecast from first 3 scans"}
ITEMS = {(p, s): make_protocol(SPLIT[s], p) for p in PROTOCOLS for s in ["cal", "test"]}

# Table 1
rows = []
for s in STUDIES:
    m = studies == s
    rows.append(dict(Trial=f"Study {s}", Lesions=int(m.sum()), **{f"Lesions >= {CFG['MIN_OBS_EVAL']} scans": int((m & (n_obs >= CFG['MIN_OBS_EVAL'])).sum())},
                     Median_scans=float(np.median(n_obs[m])), Max_followup_days=float(max(PATIENTS[i]["t"][-1] for i in np.where(m)[0]) * TU),
                     Train=int(np.isin(np.where(m)[0], SPLIT["train"]).sum()), Calibration=int(np.isin(np.where(m)[0], SPLIT["cal"]).sum()),
                     Test=int(np.isin(np.where(m)[0], SPLIT["test"]).sum())))
cohort = pd.DataFrame(rows)
tot = cohort.drop(columns=["Trial"]).sum(numeric_only=True)
tot["Median_scans"] = float(np.median(n_obs)); tot["Max_followup_days"] = cohort.Max_followup_days.max()
cohort = pd.concat([cohort, pd.DataFrame([dict(Trial="All", **tot.to_dict())])], ignore_index=True)
for _c in cohort.columns:
    if _c not in ("Trial", "Median_scans", "Max_followup_days"):
        cohort[_c] = cohort[_c].astype(int)
save_table(cohort, "T1_cohort", caption="Table 1. Cohort by trial and data split")

pd.DataFrame([dict(pid=p["pid"], patient_key=PKEY[i], study=p["study"], arm=p["arm"], split=split_of[i], n_scans=p["n"]) for i, p in enumerate(PATIENTS)]
             ).to_csv(os.path.join(DIRS["data"], "patient_split.csv"), index=False)
pd.concat([pd.DataFrame(dict(pid=p["pid"], study=p["study"], t_model=p["t"], z=p["z"], diam=p["diam"], vol=p["vol"])) for p in PATIENTS]
          ).to_csv(os.path.join(DIRS["data"], "processed_long.csv"), index=False)

# Fig 1
STUDY_COL = {s: PAL[i] for i, s in enumerate(STUDIES)}
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
bins = np.arange(2.5, n_obs.max() + 1.5)
ax[0].hist(n_obs, bins=bins, color=PAL[0], edgecolor="white", linewidth=1.5)
ax[0].axvline(CFG["MIN_OBS_EVAL"] - 0.5, color=INK2, ls="--", lw=1.2)
ax[0].set(xlabel="Scans per lesion", ylabel="Lesions", title="(a) Scans per lesion")
rng = np.random.default_rng(0)
for s in STUDIES:
    cand = [i for i in idx_eval if PATIENTS[i]["study"] == s]
    for i in rng.choice(cand, size=min(30, len(cand)), replace=False):
        ax[1].plot(PATIENTS[i]["t"] * TU, PATIENTS[i]["z"], color=STUDY_COL[s], alpha=0.45, lw=1.0)
ax[1].set(xlabel="Days from baseline scan", ylabel="log relative volume z", title="(b) Example trajectories (30 per trial)")
ax[1].legend(handles=[Line2D([], [], color=STUDY_COL[s], lw=2, label=f"Study {s}") for s in STUDIES], loc="lower left", fontsize=8)
w = 0.26
xs = np.arange(len(STUDIES))
for j, (sp, col) in enumerate([("train", PAL[0]), ("cal", PAL[1]), ("test", PAL[2])]):
    cnt = [np.isin(np.where(studies == s)[0], SPLIT[sp]).sum() for s in STUDIES]
    ax[2].bar(xs + (j - 1) * w, cnt, width=w - 0.02, color=col, label={"train": "Train", "cal": "Calibration", "test": "Test"}[sp])
ax[2].set_xticks(xs, [f"Study {s}" for s in STUDIES])
ax[2].set(ylabel="Lesions", title="(c) Split by trial")
ax[2].legend()
fig.tight_layout()
save_fig(fig, "F1_cohort_overview")

## Solver and building blocks
Fixed-step RK4 (10 days), batched over lesions, each lesion starting from its own last scan. Values at the actual scan times are interpolated from the grid. It's all torch ops, so gradients go straight through the solver.

In [ ]:
DT = CFG["DT"]

def pack(seqs, fill=0.0):
    L = max(len(s) for s in seqs)
    out = np.full((len(seqs), L), fill, dtype=np.float32)
    mask = np.zeros((len(seqs), L), dtype=np.float32)
    for i, s in enumerate(seqs):
        out[i, :len(s)] = s
        mask[i, :len(s)] = 1.0
    return torch.tensor(out), torch.tensor(mask)

def rk4_grid(rhs, z0, n_steps, t0=0.0):
    # t0 can be a float or a (B,) tensor of start times
    z = z0
    traj = [z]
    for k in range(n_steps):
        t = t0 + k * DT
        k1 = rhs(z, t)
        k2 = rhs(z + 0.5 * DT * k1, t + 0.5 * DT)
        k3 = rhs(z + 0.5 * DT * k2, t + 0.5 * DT)
        k4 = rhs(z + DT * k3, t + DT)
        z = (z + DT / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)).clamp(ZMIN, ZMAX)
        traj.append(z)
    return torch.stack(traj, 1)  # B x (n_steps+1)

def interp_grid(zg, T):
    # zg is B x (n+1) on the solver grid, T is B x L elapsed times
    n_steps = zg.shape[1] - 1
    pos = (T / DT).clamp(0, n_steps - 1e-4)
    i0 = torch.floor(pos).long().clamp(0, n_steps - 1)
    w = pos - i0.float()
    return zg.gather(1, i0) * (1 - w) + zg.gather(1, i0 + 1) * w

def as_tensor_like(t, z):
    return t.expand_as(z) if torch.is_tensor(t) else torch.full_like(z, float(t))

class MLP(nn.Module):
    def __init__(self, d_in, h, d_out=1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, h), nn.Tanh(), nn.Linear(h, h), nn.Tanh(), nn.Linear(h, d_out))
        with torch.no_grad():
            self.net[-1].weight.mul_(0.1)
            self.net[-1].bias.zero_()
    def forward(self, x):
        return self.net(x)

class NeuralG(nn.Module):
    # shared effect term. g = 1 + MLP, so at init this is just the mechanistic model
    def __init__(self, h):
        super().__init__()
        self.mlp = MLP(2, h)
    def forward(self, z, t):
        return 1.0 + self.mlp(torch.stack([z / 3.0, as_tensor_like(t, z) / 4.0], -1)).squeeze(-1)

LIB_NAMES = ["1", "z", "z^2", "z^3", "t", "t^2", "exp(-t)", "z*t", "z*exp(-t)", "t*exp(-t)"]

def library_np(z, t):
    e = np.exp(-t)
    return np.stack([np.ones_like(z), z, z ** 2, z ** 3, t, t ** 2, e, z * t, z * e, t * e], -1)

def library_torch(z, t):
    tt = as_tensor_like(t, z)
    e = torch.exp(-tt)
    return torch.stack([torch.ones_like(z), z, z ** 2, z ** 3, tt, tt ** 2, e, z * tt, z * e, tt * e], -1)

class SymbolicG(nn.Module):
    # g(z, t) = sum of the active library terms
    def __init__(self, coef, active, trainable=True):
        super().__init__()
        self.register_buffer("active", torch.tensor(active, dtype=torch.float32))
        self.coef = nn.Parameter(torch.tensor(coef, dtype=torch.float32), requires_grad=trainable)
    def forward(self, z, t):
        return (library_torch(z, t) * (self.coef * self.active)).sum(-1)

class ConstG(nn.Module):
    # ablation: g = 1, i.e. dz/dt = rho - kappa
    def forward(self, z, t):
        return torch.ones_like(z)

## Proposed model
A GRU encoder reads the observed scans and returns δ, ρ, κ and two scale parameters for each lesion. From the last scan I integrate dz/dt = ρ − κ·g_θ(z, t), where g_θ = 1 + small MLP is shared by all lesions (so training starts from the plain mechanistic model).

Loss: Laplace NLL of the future scans with a horizon-dependent scale b(h). The observation cut-off is redrawn every epoch, so the model learns to forecast rather than to fit whole curves.

The same encoder is reused for two ablations: g = 1 (mechanistic only), and a latent neural ODE with no mechanistic part.

In [ ]:
class AmortizedODE(nn.Module):
    def __init__(self, h, kind="ude", g=None):
        super().__init__()
        self.kind = kind
        self.gru = nn.GRU(3, h, batch_first=True)
        self.head = nn.Linear(h, 5)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)
        if kind == "ude":
            self.g = g if g is not None else NeuralG(h)
        elif kind == "mech":
            self.g = ConstG()
        elif kind == "node":
            self.f = MLP(4, h)
        else:
            raise ValueError(kind)

    def encode(self, T, Z, lens):
        dt = torch.cat([torch.zeros_like(T[:, :1]), T[:, 1:] - T[:, :-1]], 1)
        x = torch.stack([T / 4.0, Z / 3.0, dt / 4.0], -1)
        _, h = self.gru(pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False))
        return self.head(h[-1])

    @staticmethod
    def rho(p):
        return 2.0 * torch.tanh(p[:, 1] / 2.0)

    @staticmethod
    def kappa(p):
        return torch.exp((p[:, 2] - 1.0).clamp(-6.0, 3.0))

    def rhs_fn(self, p):
        if self.kind == "node":
            return lambda z, t: self.f(torch.stack([z / 3.0, as_tensor_like(t, z) / 4.0, p[:, 1], p[:, 2]], -1)).squeeze(-1).clamp(-20.0, 20.0)
        rho, kap = self.rho(p), self.kappa(p)
        return lambda z, t: (rho - kap * self.g(z, t)).clamp(-20.0, 20.0)

    def rollout(self, T, Z, lens, h_max):
        p = self.encode(T, Z, lens)
        li = (lens - 1).unsqueeze(1)
        t_last, z_last = T.gather(1, li)[:, 0], Z.gather(1, li)[:, 0]
        z0 = z_last + 0.5 * torch.tanh(p[:, 0])
        n_steps = int(math.ceil(float(h_max) / DT)) + 1
        zg = rk4_grid(self.rhs_fn(p), z0, n_steps, t0=t_last)
        return zg, t_last, p

    @staticmethod
    def scale(p, H):
        return nn.functional.softplus(p[:, 3:4] - 1.0 + (0.5 + torch.sigmoid(p[:, 4:5])) * torch.log(H + CFG["H0"])) + 1e-3

    def forward(self, T, Z, lens, Tq):
        li = (lens - 1).unsqueeze(1)
        H = (Tq - T.gather(1, li)).clamp_min(0.0)
        zg, _, p = self.rollout(T, Z, lens, H.max())
        return interp_grid(zg, H), self.scale(p, H), p

def seq_batch(seq_list):
    To, _ = pack([s[0] for s in seq_list]); Zo, _ = pack([s[1] for s in seq_list])
    Tq, Mq = pack([s[2] for s in seq_list]); Zq, _ = pack([s[3] for s in seq_list])
    lens = torch.tensor([len(s[0]) for s in seq_list])
    return To, Zo, lens, Tq, Zq, Mq

def random_cut_batches(idx, rng, batch):
    # fresh cut-off k in [3, n-1] for each lesion every time this is called
    perm = rng.permutation(idx)
    for b in range(0, len(perm), batch):
        seqs = []
        for i in perm[b:b + batch]:
            p = PATIENTS[i]
            k = int(rng.integers(3, p["n"]))
            seqs.append((p["t"][:k], p["z"][:k], p["t"][k:], p["z"][k:]))
        yield seq_batch(seqs)

TRAIN_SEQ = [i for i in SPLIT["train"] if PATIENTS[i]["n"] >= 4]

def train_amortized(model, seed, epochs, lr, label):
    set_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-4)
    rng = np.random.default_rng(seed)
    hist, t0 = [], time.time()
    for ep in range(epochs):
        model.train()
        tot_nll = tot_ae = cnt = 0.0
        for To, Zo, lens, Tq, Zq, Mq in random_cut_batches(TRAIN_SEQ, rng, CFG["BATCH"]):
            opt.zero_grad()
            zp, b, p = model(To, Zo, lens, Tq)
            ae = (zp - Zq).abs()
            nll = ((ae / b + torch.log(2 * b)) * Mq).sum()
            loss = nll / Mq.sum() + 1e-3 * (p ** 2).mean()
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss in {label} at epoch {ep}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tot_nll += float(nll); tot_ae += float((ae * Mq).sum()); cnt += float(Mq.sum())
        sched.step()
        hist.append(dict(epoch=ep, nll=tot_nll / cnt, mae=tot_ae / cnt))
        if ep % max(1, epochs // 5) == 0 or ep == epochs - 1:
            log(f"  {label} seed={seed} epoch {ep:4d}/{epochs}  NLL={hist[-1]['nll']:.4f}  train MAE(z)={hist[-1]['mae']:.4f}")
    log(f"  {label} seed={seed} trained in {time.time() - t0:.1f}s")
    return pd.DataFrame(hist), time.time() - t0

def run_amortized(model, items):
    model.eval()
    seqs = [(PATIENTS[i]["t"][o], PATIENTS[i]["z"][o], PATIENTS[i]["t"][tg], PATIENTS[i]["z"][tg]) for i, o, tg in items]
    with torch.no_grad():
        To, Zo, lens, Tq, _, _ = seq_batch(seqs)
        zp, b, _ = model(To, Zo, lens, Tq)
    zp, b = zp.numpy(), b.numpy()
    return ([zp[k, :len(tg)].astype(float) for k, (_, _, tg) in enumerate(items)],
            [b[k, :len(tg)].astype(float) for k, (_, _, tg) in enumerate(items)])

## Baselines
LOCF, the growth laws from Laleh et al. plus Stein, a per-lesion GP, Stein-EB (population Stein model + per-lesion MAP, basically an NLME), UDE-MAP (same ODE but fitted the usual way: curve fit, then MAP per lesion) and a GRU forecaster trained on the same cut-offs.

In [ ]:
def z_exponential(th, t):
    return th[0] + th[1] * t

def z_logistic(th, t):
    y0, a, K = np.exp(th[0]), th[1], np.exp(th[2])
    ea = np.exp(np.clip(a * t, -30, 30))
    return np.log(K) + th[0] + a * t - np.log(np.maximum(K + y0 * (ea - 1.0), 1e-8))

def z_gompertz(th, t):
    return th[2] + (th[0] - th[2]) * np.exp(np.clip(-th[1] * t, -30, 30))

def z_bertalanffy(th, t):
    y0, A, B = np.exp(th[0]), np.exp(th[1]), np.exp(th[2])
    w = A / B + (y0 ** (1 / 3) - A / B) * np.exp(np.clip(-B * t / 3.0, -30, 30))
    return 3.0 * np.log(np.maximum(w, 1e-4))

def z_gen_bertalanffy(th, t):
    y0, A, B = np.exp(th[0]), np.exp(th[1]), np.exp(th[2])
    m = 0.01 + 0.98 / (1.0 + np.exp(-th[3]))
    w = A / B + (y0 ** (1 - m) - A / B) * np.exp(np.clip(-(1 - m) * B * t, -30, 30))
    return np.log(np.maximum(w, 1e-6)) / (1 - m)

def z_stein(th, t):
    d, g = np.exp(th[1]), np.exp(th[2])
    f = np.exp(np.clip(-d * t, -30, 30)) + np.exp(np.clip(g * t, -30, 15)) - 1.0
    return th[0] + np.log(np.maximum(f, 1e-5))

CLASSICAL = {
    "Exponential": (z_exponential, [[0, 0.0], [0, -0.5], [0, 0.5]], ([-5, -5], [5, 5])),
    "Logistic": (z_logistic, [[0, 0.0, 1.0], [0, -0.5, 0.5], [0, 0.5, 2.0]], ([-5, -5, -5], [5, 5, 6])),
    "Gompertz": (z_gompertz, [[0, 0.5, 0.0], [0, 0.2, -1.0], [0, 1.0, 1.0]], ([-5, -5, -8], [5, 5, 6])),
    "Bertalanffy": (z_bertalanffy, [[0, 0.0, 0.0], [0, -1.0, 0.0], [0, 0.5, 1.0]], ([-5, -6, -6], [5, 4, 4])),
    "Gen. Bertalanffy": (z_gen_bertalanffy, [[0, 0.0, 0.0, 0.0], [0, -1.0, 0.0, -1.0], [0, 0.5, 1.0, 1.0]], ([-5, -6, -6, -6], [5, 4, 4, 6])),
    "Stein": (z_stein, [[0, -1.0, -2.0], [0, 0.0, -1.0], [0, -2.0, -3.0]], ([-5, -8, -8], [5, 3, 3])),
}

def fit_classical(fun, starts, bounds, t, z):
    best = None
    lo, hi = np.array(bounds[0], float), np.array(bounds[1], float)
    for x0 in starts:
        x0 = np.clip(np.array(x0, float), lo + 1e-6, hi - 1e-6)
        x0[0] = np.clip(z[0], lo[0] + 1e-6, hi[0] - 1e-6)
        res_f = lambda th, x0=x0: np.concatenate([np.clip(fun(th, t), ZMIN, ZMAX) - z, 1e-2 * (th - x0)])
        try:
            r = least_squares(res_f, x0, bounds=(lo, hi), method="trf", max_nfev=400)
            if best is None or r.cost < best.cost:
                best = r
        except Exception:
            continue
    return best.x if best is not None else np.clip(np.array(starts[0], float), lo + 1e-6, hi - 1e-6)

def run_classical(name, items):
    fun, starts, bounds = CLASSICAL[name]
    out = []
    for i, o, tg in items:
        p = PATIENTS[i]
        th = fit_classical(fun, starts, bounds, p["t"][o], p["z"][o])
        out.append(np.clip(fun(th, p["t"][tg]), ZMIN, ZMAX))
    return out

def run_gp(items):
    out = []
    for i, o, tg in items:
        p = PATIENTS[i]
        k = ConstantKernel(1.0, (1e-2, 1e2)) * RBF(2.0, (0.3, 20.0)) + WhiteKernel(0.05, (1e-4, 1.0))
        gp = GaussianProcessRegressor(kernel=k, normalize_y=False, n_restarts_optimizer=2, random_state=0)
        gp.fit(p["t"][o][:, None], p["z"][o])
        out.append(np.clip(gp.predict(p["t"][tg][:, None]), ZMIN, ZMAX))
    return out

def run_locf(items):
    return [np.full(len(tg), PATIENTS[i]["z"][o[-1]]) for i, o, tg in items]

# population fit + per-lesion MAP (Stein-EB and the UDE-MAP ablation)
class BaseMAP(nn.Module):
    closed_form = False
    prior_mu0 = [0.0, 0.0, 0.0]
    prior_sd0 = [0.5, 1.5, 1.5]
    def init_raw(self, B, z_first, mu=None):
        mu = torch.tensor(self.prior_mu0) if mu is None else mu
        raw = mu.repeat(B, 1).clone()
        raw[:, 0] = z_first
        return raw
    def predict(self, raw, T):
        if self.closed_form:
            return self.closed(raw, T).clamp(ZMIN, ZMAX)
        n_steps = int(math.ceil(float(T.max()) / DT)) + 1
        zg = rk4_grid(lambda z, t: self.rhs(z, t, raw).clamp(-20.0, 20.0), raw[:, 0], n_steps)
        return interp_grid(zg, T)

class UDEMAP(BaseMAP):
    def __init__(self, h):
        super().__init__()
        self.g = NeuralG(h)
    def rhs(self, z, t, raw):
        return 4.0 * torch.tanh(raw[:, 1] / 4.0) - torch.exp(raw[:, 2].clamp(-6.0, 3.0)) * self.g(z, t)

class SteinEB(BaseMAP):
    closed_form = True
    prior_mu0 = [0.0, -1.0, -2.0]
    def closed(self, raw, T):
        d = torch.exp(raw[:, 1:2].clamp(-8.0, 3.0))
        gr = torch.exp(raw[:, 2:3].clamp(-8.0, 3.0))
        f = torch.exp(-(d * T).clamp(max=30.0)) + torch.exp((gr * T).clamp(max=15.0)) - 1.0
        return raw[:, 0:1] + torch.log(f.clamp_min(1e-5))

def fit_population(model, idx, epochs, seed, label):
    set_seed(seed)
    T, M = pack([PATIENTS[i]["t"] for i in idx]); Z, _ = pack([PATIENTS[i]["z"] for i in idx])
    zf = torch.tensor([PATIENTS[i]["z"][0] for i in idx], dtype=torch.float32)
    raw = nn.Parameter(model.init_raw(len(idx), zf))
    log_sigma = nn.Parameter(torch.tensor(math.log(0.3)))
    mu0, sd0 = torch.tensor(model.prior_mu0), torch.tensor(model.prior_sd0)
    groups = [{"params": [raw, log_sigma], "lr": CFG["LR_PAT"]}]
    net = [p for p in model.parameters() if p.requires_grad]
    if net:
        groups.append({"params": net, "lr": CFG["LR_NET"]})
    opt = torch.optim.Adam(groups)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-4)
    n_tot = M.sum()
    hist, t0 = [], time.time()
    for ep in range(epochs):
        opt.zero_grad()
        pred = model.predict(raw, T)
        sse = (((pred - Z) ** 2) * M).sum()
        loss = (sse / (2 * torch.exp(2 * log_sigma)) + n_tot * log_sigma + ((((raw - mu0) / sd0) ** 2) / 2).sum()) / n_tot
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net + [raw], 10.0)
        opt.step(); sched.step()
        hist.append(dict(epoch=ep, mae=float(((pred - Z).abs() * M).sum() / n_tot)))
        if ep % max(1, epochs // 5) == 0 or ep == epochs - 1:
            log(f"  {label} seed={seed} epoch {ep:4d}/{epochs}  fit MAE(z)={hist[-1]['mae']:.4f}")
    raw_d = raw.detach()
    return dict(mu=raw_d.mean(0), sd=raw_d.std(0).clamp_min(0.05), sigma=float(torch.exp(log_sigma.detach())),
                hist=pd.DataFrame(hist), seconds=time.time() - t0)

def adapt_predict(model, items, prior):
    # MAP on the observed window under the EB prior, then forecast
    for p in model.parameters():
        p.requires_grad_(False)
    To, Mo = pack([PATIENTS[i]["t"][o] for i, o, _ in items]); Zo, _ = pack([PATIENTS[i]["z"][o] for i, o, _ in items])
    Tt, _ = pack([PATIENTS[i]["t"][tg] for i, _, tg in items])
    zf = torch.tensor([PATIENTS[i]["z"][o[0]] for i, o, _ in items], dtype=torch.float32)
    mu, sd, sig2 = prior["mu"], prior["sd"], prior["sigma"] ** 2
    raw = nn.Parameter(model.init_raw(len(items), zf, mu=mu))
    opt = torch.optim.Adam([raw], lr=CFG["LR_ADAPT"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["EPOCHS_ADAPT"], eta_min=1e-3)
    for _ in range(CFG["EPOCHS_ADAPT"]):
        opt.zero_grad()
        pred = model.predict(raw, To)
        loss = (((pred - Zo) ** 2) * Mo).sum() / (2 * sig2) + ((((raw - mu) / sd) ** 2) / 2).sum()
        loss.backward()
        opt.step(); sched.step()
    with torch.no_grad():
        pt = model.predict(raw.detach(), Tt).numpy()
    return [pt[k, :len(tg)].astype(float) for k, (_, _, tg) in enumerate(items)]

# GRU baseline
class GRUForecaster(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.gru = nn.GRU(3, h, batch_first=True)
        self.dec = MLP(h + 2, h)
    def forward(self, T, Z, lens, Tq):
        dt = torch.cat([torch.zeros_like(T[:, :1]), T[:, 1:] - T[:, :-1]], 1)
        x = torch.stack([T / 4.0, Z / 3.0, dt / 4.0], -1)
        _, h = self.gru(pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False))
        h = h[-1]
        li = (lens - 1).unsqueeze(1)
        t_last, z_last = T.gather(1, li), Z.gather(1, li)
        inp = torch.cat([h.unsqueeze(1).expand(-1, Tq.shape[1], -1), ((Tq - t_last) / 4.0).unsqueeze(-1),
                         (z_last / 3.0).expand_as(Tq).unsqueeze(-1)], -1)
        return (z_last + self.dec(inp).squeeze(-1)).clamp(ZMIN, ZMAX)

def train_gru(seed):
    set_seed(seed)
    model = GRUForecaster(CFG["HIDDEN"])
    opt = torch.optim.Adam(model.parameters(), lr=CFG["AMORT_LR"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["AMORT_EPOCHS"], eta_min=1e-4)
    rng = np.random.default_rng(seed)
    hist, t0 = [], time.time()
    for ep in range(CFG["AMORT_EPOCHS"]):
        tot = cnt = 0.0
        for To, Zo, lens, Tq, Zq, Mq in random_cut_batches(TRAIN_SEQ, rng, CFG["BATCH"]):
            opt.zero_grad()
            ae = ((model(To, Zo, lens, Tq) - Zq).abs() * Mq).sum()
            (ae / Mq.sum()).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tot += float(ae); cnt += float(Mq.sum())
        sched.step()
        hist.append(dict(epoch=ep, mae=tot / cnt))
        if ep % max(1, CFG["AMORT_EPOCHS"] // 5) == 0 or ep == CFG["AMORT_EPOCHS"] - 1:
            log(f"  GRU seed={seed} epoch {ep:4d}/{CFG['AMORT_EPOCHS']}  train MAE(z)={hist[-1]['mae']:.4f}")
    return model, pd.DataFrame(hist), time.time() - t0

def run_gru(model, items):
    model.eval()
    seqs = [(PATIENTS[i]["t"][o], PATIENTS[i]["z"][o], PATIENTS[i]["t"][tg], PATIENTS[i]["z"][tg]) for i, o, tg in items]
    with torch.no_grad():
        To, Zo, lens, Tq, _, _ = seq_batch(seqs)
        pred = model(To, Zo, lens, Tq).numpy()
    return [pred[k, :len(tg)].astype(float) for k, (_, _, tg) in enumerate(items)]

## Train everything
Classical fits and the GP run once; the neural models run for every seed. This is the slow part.

In [ ]:
PROPOSED, SYMB, LNODE, MECH, UDEMAP_NAME = "UDE (proposed)", "Symbolic UDE", "Latent neural ODE", "Mechanistic (no NN)", "UDE-MAP (no amortization)"
KEYS = [(p, s) for p in PROTOCOLS for s in ["cal", "test"]]
PRED, SCALE, HIST, MODELS = {}, {}, {}, {}

def predict_all(fn):
    return {k: fn(ITEMS[k]) for k in KEYS}

def predict_joint(fn):
    # lesions are independent, so run all protocol/set combos in one batch and split afterwards
    joint = [it for k in KEYS for it in ITEMS[k]]
    out = fn(joint)
    res, pos = {}, 0
    for k in KEYS:
        res[k] = out[pos:pos + len(ITEMS[k])]
        pos += len(ITEMS[k])
    return res

def store_amortized(name, seed, model):
    both = predict_joint(lambda it: list(zip(*run_amortized(model, it))))
    PRED[(name, seed)] = {k: [a for a, _ in v] for k, v in both.items()}
    SCALE[(name, seed)] = {k: [b for _, b in v] for k, v in both.items()}

t0 = time.time()
PRED[("Persistence (LOCF)", 0)] = predict_all(run_locf)
for name in CLASSICAL:
    ts = time.time()
    PRED[(name, 0)] = predict_all(lambda it, n=name: run_classical(n, it))
    log(f"Classical {name} fitted in {time.time() - ts:.1f}s")
ts = time.time()
PRED[("Gaussian process", 0)] = predict_all(run_gp)
log(f"Gaussian process fitted in {time.time() - ts:.1f}s")
TIMINGS["classical_and_GP_s"] = time.time() - t0

SEEDS = list(range(CFG["SEED"], CFG["SEED"] + CFG["N_SEEDS"]))
for seed in SEEDS:
    log(f"===== Seed {seed} =====")
    for name, kind in [(PROPOSED, "ude"), (LNODE, "node"), (MECH, "mech")]:
        set_seed(seed)
        m = AmortizedODE(CFG["HIDDEN"], kind)
        h, secs = train_amortized(m, seed, CFG["AMORT_EPOCHS"], CFG["AMORT_LR"], name)
        HIST[(name, seed)], MODELS[(name, seed)] = h, m
        TIMINGS[f"{name} seed{seed} train_s"] = secs
        store_amortized(name, seed, m)
        torch.save(m.state_dict(), os.path.join(DIRS["models"], f"{kind}_amortized_seed{seed}.pt"))
    gm, gh, gs = train_gru(seed)
    HIST[("GRU", seed)], MODELS[("GRU", seed)] = gh, gm
    TIMINGS[f"GRU seed{seed} train_s"] = gs
    PRED[("GRU", seed)] = predict_joint(lambda it, m=gm: run_gru(m, it))
    torch.save(gm.state_dict(), os.path.join(DIRS["models"], f"GRU_seed{seed}.pt"))

# Stein-EB and UDE-MAP: first seed only (UDE-MAP is slow)
for name, ctor in [("Stein-EB", SteinEB), (UDEMAP_NAME, lambda: UDEMAP(CFG["HIDDEN"]))]:
    ts = time.time()
    set_seed(SEEDS[0])
    m = ctor()
    pop = fit_population(m, SPLIT["train"], CFG["EPOCHS_POP"], SEEDS[0], name)
    PRED[(name, SEEDS[0])] = predict_joint(lambda it, m=m, pop=pop: adapt_predict(m, it, pop))
    HIST[(name, SEEDS[0])], MODELS[(name, SEEDS[0])] = pop["hist"], m
    TIMINGS[f"{name} fit_and_adapt_s"] = time.time() - ts
    torch.save(dict(state=m.state_dict(), mu=pop["mu"], sd=pop["sd"], sigma=pop["sigma"]),
               os.path.join(DIRS["models"], f"{name.split(' ')[0].replace('-', '_')}.pt"))
log("All models finished")

## Symbolic recovery
Sample g_θ (seed 0) along forecast trajectories of the training lesions and run STLSQ over a 10-term library. The rule for picking the law was fixed beforehand: the sparsest one that keeps at least 95% of the full-library R². That law then replaces the network and the model is fine-tuned briefly.

In [ ]:
ts = time.time()
ude0 = MODELS[(PROPOSED, SEEDS[0])]
ude0.eval()
rng = np.random.default_rng(123)
Zs_list, Ts_list = [], []
with torch.no_grad():
    for _ in range(3):
        seqs, t_end = [], []
        for i in TRAIN_SEQ:
            p = PATIENTS[i]
            k = int(rng.integers(3, p["n"]))
            seqs.append((p["t"][:k], p["z"][:k], p["t"][k:], p["z"][k:]))
            t_end.append(p["t"][-1])
        To, Zo, lens, Tq, _, _ = seq_batch(seqs)
        t_end = torch.tensor(t_end, dtype=torch.float32)
        li = (lens - 1).unsqueeze(1)
        H = t_end - To.gather(1, li)[:, 0]
        zg, t_last, _ = ude0.rollout(To, Zo, lens, H.max())
        steps = torch.arange(zg.shape[1], dtype=torch.float32) * DT
        valid = steps[None, :] <= H[:, None]
        Zs_list.append(zg[valid].numpy()); Ts_list.append((t_last[:, None] + steps[None, :])[valid].numpy())
Zs, Ts = np.concatenate(Zs_list).astype(float), np.concatenate(Ts_list).astype(float)
sel = rng.choice(len(Zs), size=min(30000, len(Zs)), replace=False)
Zs, Ts = Zs[sel], Ts[sel]
with torch.no_grad():
    Gs = ude0.g(torch.tensor(Zs, dtype=torch.float32), torch.tensor(Ts, dtype=torch.float32)).numpy().astype(float)
log(f"Sampled {len(Zs)} states of the learned law: z in [{Zs.min():.2f}, {Zs.max():.2f}], t in [0, {Ts.max() * TU:.0f}] days")

Theta = library_np(Zs, Ts)
col_scale = Theta.std(0)
col_scale[0] = 1.0
col_scale[col_scale < 1e-12] = 1.0
Th = Theta / col_scale

def stlsq(A, y, thr, n_iter=20, ridge=1e-6):
    active = np.ones(A.shape[1], bool)
    coef = np.zeros(A.shape[1])
    for _ in range(n_iter):
        if not active.any():
            break
        Aa = A[:, active]
        c = np.linalg.solve(Aa.T @ Aa + ridge * np.eye(active.sum()), Aa.T @ y)
        coef[:] = 0.0
        coef[active] = c
        new_active = np.abs(coef) >= thr
        if (new_active == active).all():
            break
        active = new_active
    return coef, active

def r2(y, yhat):
    return 1 - np.sum((y - yhat) ** 2) / max(np.sum((y - y.mean()) ** 2), 1e-12)

pareto = []
for thr in np.concatenate([[0.0], np.logspace(-4, 0.5, 60)]):
    c, a = stlsq(Th, Gs, thr)
    if a.sum() == 0:
        continue
    pareto.append(dict(threshold=thr, n_terms=int(a.sum()), R2=r2(Gs, Th @ c), coef=c.copy(), active=a.copy()))
par = pd.DataFrame(pareto).sort_values(["n_terms", "R2"], ascending=[True, False]).drop_duplicates("n_terms").reset_index(drop=True)
r2_full = float(par.R2.max())
chosen = par[par.R2 >= CFG["STLSQ_R2_FRACTION"] * r2_full].sort_values("n_terms").iloc[0]
coef_sym = chosen["coef"] / col_scale
active_sym = chosen["active"].astype(float)

def fmt_law(coef, active):
    terms = [(f"{coef[k]:+.4f}" if LIB_NAMES[k] == "1" else f"{coef[k]:+.4f}*{LIB_NAMES[k]}") for k in range(len(coef)) if active[k] > 0]
    return "g(z,t) = " + " ".join(terms)

law_stlsq = fmt_law(coef_sym, active_sym)
log(f"STLSQ selected {int(chosen['n_terms'])} terms, R2={chosen['R2']:.4f} (full library R2={r2_full:.4f}): {law_stlsq}")
save_table(pd.DataFrame([dict(n_terms=int(r.n_terms), R2=round(float(r.R2), 5), law=fmt_law(r.coef / col_scale, r.active.astype(float)))
                         for r in par.itertuples()]),
           "T5a_symbolic_pareto", caption="Table 5a. STLSQ Pareto front: sparsity versus fidelity to the learned network")

# swap the network for the recovered law, keep the encoder, short fine-tune
symb = copy.deepcopy(ude0)
symb.g = SymbolicG(coef_sym, active_sym, trainable=True)
h, secs = train_amortized(symb, SEEDS[0], CFG["SYMB_EPOCHS"], 1e-3, SYMB)
HIST[(SYMB, SEEDS[0])], MODELS[(SYMB, SEEDS[0])] = h, symb
store_amortized(SYMB, SEEDS[0], symb)
coef_ref = symb.g.coef.detach().numpy() * active_sym
law_refined = fmt_law(coef_ref, active_sym)
log(f"Refined law: {law_refined}")
torch.save(dict(state=symb.state_dict(), coef=coef_ref, active=active_sym, library=LIB_NAMES),
           os.path.join(DIRS["models"], "symbolic_ude_seed0.pt"))
TIMINGS["symbolic_recovery_s"] = time.time() - ts

# rho / kappa on the P2 test lesions (Fig 10)
it = ITEMS[("P2", "test")]
seqs = [(PATIENTS[i]["t"][o], PATIENTS[i]["z"][o], PATIENTS[i]["t"][tg], PATIENTS[i]["z"][tg]) for i, o, tg in it]
with torch.no_grad():
    To, Zo, lens, _, _, _ = seq_batch(seqs)
    pp = symb.encode(To, Zo, lens)
    rho_te, kap_te = AmortizedODE.rho(pp).numpy(), AmortizedODE.kappa(pp).numpy()
PARAMS_TE = pd.DataFrame(dict(pid=[PATIENTS[i]["pid"] for i, _, _ in it], study=[PATIENTS[i]["study"] for i, _, _ in it], rho=rho_te, kappa=kap_te))
PARAMS_TE.to_csv(os.path.join(DIRS["predictions"], "symbolic_ude_patient_parameters_P2_test.csv"), index=False)
law_tab = pd.DataFrame([
    dict(item="Model", value="dz/dt = rho_i - kappa_i * g(z,t);  z = log relative volume;  t = time since baseline (units of 100 days)"),
    dict(item="Law selected by STLSQ", value=law_stlsq),
    dict(item="Law after fine-tuning inside the ODE", value=law_refined),
    dict(item="R2 of selected law vs network", value=f"{chosen['R2']:.4f}"),
    dict(item="R2 of full 10-term library vs network", value=f"{r2_full:.4f}"),
    dict(item="rho_i on P2 test lesions, median [IQR] per 100 days", value=f"{np.median(rho_te):.3f} [{np.percentile(rho_te, 25):.3f}, {np.percentile(rho_te, 75):.3f}]"),
    dict(item="kappa_i on P2 test lesions, median [IQR]", value=f"{np.median(kap_te):.3f} [{np.percentile(kap_te, 25):.3f}, {np.percentile(kap_te, 75):.3f}]"),
])
save_table(law_tab, "T5b_discovered_law", caption="Table 5b. Discovered tumor response law")

## Accuracy and tests
Main metric: MAE of z per lesion, averaged over test lesions (so each lesion counts once), with a bootstrap CI. Neural models are mean ± SD over seeds; the tests and everything after use seed 0.

In [ ]:
MAIN_ORDER = ["Persistence (LOCF)", "Exponential", "Logistic", "Gompertz", "Bertalanffy", "Gen. Bertalanffy", "Stein",
              "Gaussian process", "Stein-EB", "GRU", LNODE, PROPOSED, SYMB]
ABLATION = [MECH, LNODE, UDEMAP_NAME, PROPOSED, SYMB]
REF = "Persistence (LOCF)"

rows = []
for (m, s), pr_dict in PRED.items():
    for (proto, sset), preds in pr_dict.items():
        scales = SCALE.get((m, s), {}).get((proto, sset))
        for n_, ((i, o, tg), zp) in enumerate(zip(ITEMS[(proto, sset)], preds)):
            p = PATIENTS[i]
            t_last = p["t"][o[-1]]
            for j, k in enumerate(tg):
                rows.append((m, s, proto, sset, p["pid"], i, p["study"], j, p["t"][k], p["t"][k] - t_last, p["z"][k], float(zp[j]),
                             float(scales[n_][j]) if scales is not None else np.nan))
LP = pd.DataFrame(rows, columns=["model", "seed", "protocol", "set", "pid", "pidx", "study", "j", "t", "horizon", "z_true", "z_pred", "scale"])
LP["abs_err"] = (LP.z_pred - LP.z_true).abs()
LP.to_csv(os.path.join(DIRS["predictions"], "all_predictions_long.csv.gz"), index=False, compression="gzip")
log(f"Prediction table: {len(LP)} rows")

def first_seed(model):
    return min(s for (m, s) in PRED if m == model)

def sub_of(model, proto, sset="test", seed=None):
    seed = first_seed(model) if seed is None else seed
    return LP[(LP.model == model) & (LP.seed == seed) & (LP.protocol == proto) & (LP.set == sset)]

def per_patient_mae(sub):
    return sub.groupby("pidx").abs_err.mean()

def boot_ci(x, n=CFG["N_BOOT"], seed=0):
    r = np.random.default_rng(seed)
    x = np.asarray(x)
    b = [x[r.integers(0, len(x), len(x))].mean() for _ in range(n)]
    return np.percentile(b, 2.5), np.percentile(b, 97.5)

def metrics(sub):
    e = sub.z_pred - sub.z_true
    rel = (np.exp(sub.z_pred) - np.exp(sub.z_true)).abs().groupby(sub.pidx).mean()
    return dict(MAE_z=per_patient_mae(sub).mean(), MedAE_z=float(np.median(np.abs(e))), RMSE_z=math.sqrt(float(np.mean(e ** 2))),
                R2_z=1 - float(np.sum(e ** 2)) / float(np.sum((sub.z_true - sub.z_true.mean()) ** 2)), MAE_relvol=rel.mean())

def results_table(models, proto):
    out = []
    for m in models:
        seeds = sorted(s for (mm, s) in PRED if mm == m)
        per_seed = [metrics(sub_of(m, proto, seed=s)) for s in seeds]
        lo, hi = boot_ci(per_patient_mae(sub_of(m, proto)).values)
        row = dict(Model=m, Seeds=len(seeds))
        for k in per_seed[0]:
            vals = np.array([d[k] for d in per_seed])
            row[k] = vals.mean()
            row[k + "_sd"] = vals.std(ddof=1) if len(vals) > 1 else np.nan
        row["MAE_z_CI95"] = f"[{lo:.3f}, {hi:.3f}]"
        out.append(row)
    return pd.DataFrame(out)

RES = {}
for pr in PROTOCOLS:
    RES[pr] = results_table(MAIN_ORDER, pr)
    save_table(RES[pr][["Model", "Seeds", "MAE_z", "MAE_z_sd", "MAE_z_CI95", "MedAE_z", "RMSE_z", "R2_z", "MAE_relvol"]].round(4),
               f"T2_{pr}_forecast_accuracy", caption=f"Table 2 ({pr}). Test-set forecast accuracy, {PROTO_DESC[pr]}")

def fmt_p(p):
    return "<0.0001" if p < 1e-4 else f"{p:.4f}"

def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for r, idx in enumerate(order):
        running = max(running, (len(p) - r) * p[idx])
        adj[idx] = min(1.0, running)
    return adj

test_rows = []
for pr in PROTOCOLS:
    ref = per_patient_mae(sub_of(PROPOSED, pr))
    tr = []
    for m in MAIN_ORDER:
        if m == PROPOSED:
            continue
        other = per_patient_mae(sub_of(m, pr))
        common = ref.index.intersection(other.index)
        d = other[common] - ref[common]
        try:
            p = float(wilcoxon(other[common], ref[common]).pvalue)
        except ValueError:
            p = 1.0
        tr.append(dict(Protocol=pr, Comparator=m, MAE_comparator=other[common].mean(), MAE_UDE=ref[common].mean(),
                       Diff_comparator_minus_UDE=d.mean(), Lesions_UDE_better_pct=100 * (d > 0).mean(), p_raw=p))
    t = pd.DataFrame(tr)
    t["p_holm"] = holm(t.p_raw.values)
    test_rows.append(t)
TESTS = pd.concat(test_rows, ignore_index=True)
TESTS_OUT = TESTS.copy()
for _c in ["p_raw", "p_holm"]:
    TESTS_OUT[_c] = TESTS_OUT[_c].map(fmt_p)
save_table(TESTS_OUT.round(5), "T3_wilcoxon_vs_UDE", caption="Table 3. Paired two-sided Wilcoxon tests of per-lesion MAE against the UDE (Holm corrected within protocol)")

# non-inferiority vs LOCF, one-sided (H1: diff < margin)
ni_rows = []
for pr in PROTOCOLS:
    ref = per_patient_mae(sub_of(REF, pr))
    for m in [PROPOSED, SYMB, LNODE, "GRU", "Gen. Bertalanffy", "Stein-EB"]:
        mm = per_patient_mae(sub_of(m, pr))
        common = ref.index.intersection(mm.index)
        d = mm[common] - ref[common]
        lo, hi = boot_ci(d.values)
        try:
            p = float(wilcoxon(d - CFG["NI_MARGIN"], alternative="less").pvalue)
        except ValueError:
            p = 1.0
        ni_rows.append(dict(Protocol=pr, Model=m, Mean_diff_vs_LOCF=d.mean(), CI95_low=lo, CI95_high=hi, Margin=CFG["NI_MARGIN"],
                            p_noninferiority=p, Non_inferior=bool(p < 0.05 and hi < CFG["NI_MARGIN"])))
NI = pd.DataFrame(ni_rows)
NI_OUT = NI.copy()
NI_OUT["p_noninferiority"] = NI_OUT["p_noninferiority"].map(fmt_p)
save_table(NI_OUT.round(5), "T3b_noninferiority_vs_persistence", caption=f"Table 3b. Non-inferiority to persistence (margin {CFG['NI_MARGIN']} in log relative volume)")

abl = []
for pr in PROTOCOLS:
    t = results_table(ABLATION, pr)
    t.insert(0, "Protocol", pr)
    abl.append(t[["Protocol", "Model", "Seeds", "MAE_z", "MAE_z_sd", "MAE_z_CI95", "RMSE_z", "R2_z"]])
save_table(pd.concat(abl, ignore_index=True).round(4), "T8_ablation", caption="Table 8. Ablation study (test set)")

### How far, and which way, do models move away from LOCF?
d = ẑ − z_last against the realised change r = z − z_last. Spearman on lesion means, and sign agreement for |d| > 0.1. A model that just adds noise to LOCF should give ~0 correlation and ~50% sign agreement.

In [ ]:
from scipy.stats import spearmanr
dev_rows = []
for pr in PROTOCOLS:
    base = sub_of(REF, pr)[["pidx", "j", "z_pred"]].rename(columns={"z_pred": "z_last"})
    for m in MAIN_ORDER:
        if m == REF:
            continue
        sub = sub_of(m, pr).merge(base, on=["pidx", "j"])
        sub["d"] = sub.z_pred - sub.z_last
        sub["r"] = sub.z_true - sub.z_last
        les = sub.groupby("pidx")[["d", "r"]].mean()
        rho_s, p_s = spearmanr(les.d, les.r)
        big = sub[sub.d.abs() > 0.1]
        dev_rows.append(dict(Protocol=pr, Model=m, Mean_abs_departure=sub.d.abs().mean(), Spearman_departure_vs_change=rho_s, p=fmt_p(p_s),
                             Sign_agreement_pct=100 * float((np.sign(big.d) == np.sign(big.r)).mean()) if len(big) else np.nan,
                             Points_with_departure_gt_0_1=len(big)))
DEV = pd.DataFrame(dev_rows)
save_table(DEV.round(4), "T10_departure_from_persistence", caption="Table 10. Size and direction of each model's departure from persistence (test set)")

# per-arm MAE (arm codes as in the raw file)
arm_rows = []
for pr in PROTOCOLS:
    for m in [REF, "Gen. Bertalanffy", "GRU", PROPOSED]:
        sub = sub_of(m, pr).copy()
        sub["arm"] = [f"Study {PATIENTS[i]['study']} arm {PATIENTS[i]['arm']}" for i in sub.pidx]
        for a, g in sub.groupby("arm"):
            arm_rows.append(dict(Protocol=pr, Arm=a, Model=m, n_lesions=g.pidx.nunique(), MAE_z=per_patient_mae(g).mean()))
ARM = pd.DataFrame(arm_rows).pivot_table(index=["Protocol", "Arm"], columns="Model", values="MAE_z").reset_index()
_n = pd.DataFrame(arm_rows).drop_duplicates(["Protocol", "Arm"]).set_index(["Protocol", "Arm"]).n_lesions
ARM.insert(2, "n_lesions", [_n[(r.Protocol, r.Arm)] for r in ARM.itertuples()])
save_table(ARM.round(4), "T11_per_arm_mae", caption="Table 11. Test MAE by trial arm (arm codes as in the source data)")

## Conformal bands
Score for each calibration lesion = max over its future scans of |ẑ − z| / b. Taking the max makes the lesion (not the scan) the exchangeable unit, so the band covers the whole future trajectory with probability ≥ 1 − α. Pooled pointwise calibration is included for comparison only; it doesn't give that guarantee.

In [ ]:
def score_scale(sub, mode):
    if mode == "learned":
        return sub.scale.values
    return np.sqrt(sub.horizon.values + CFG["H0"])

def conformal_eval(model, proto, alpha, scale_mode=None, calib="trajectory"):
    if scale_mode is None:
        scale_mode = "learned" if (model, first_seed(model)) in SCALE else "horizon"
    cal, te = sub_of(model, proto, "cal").copy(), sub_of(model, proto, "test").copy()
    cal["s"] = cal.abs_err.values / score_scale(cal, scale_mode)
    S = cal.groupby("pidx").s.max().values if calib == "trajectory" else cal.s.values
    n = len(S)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    q = np.inf if k > n else float(np.sort(S)[k - 1])
    half = q * score_scale(te, scale_mode)
    te["lo"], te["hi"] = te.z_pred - half, te.z_pred + half
    te["covered"] = (te.z_true >= te.lo) & (te.z_true <= te.hi)
    return dict(q=q, point_cov=te.covered.mean(), traj_cov=te.groupby("pidx").covered.all().mean(),
                width=float((2 * half).mean()), fold=float(np.median(np.exp(half)))), te

CONF_MODELS = ["Persistence (LOCF)", "Gen. Bertalanffy", "Stein", "Gaussian process", "Stein-EB", "GRU", LNODE, MECH, PROPOSED, SYMB]
crow = []
for pr in PROTOCOLS:
    for m in CONF_MODELS:
        variants = [(None, "trajectory")]
        if m == PROPOSED:
            variants += [("horizon", "trajectory"), (None, "pointwise")]
        for sm, cb in variants:
            r, _ = conformal_eval(m, pr, CFG["ALPHA"], sm, cb)
            smode = sm or ("learned" if (m, first_seed(m)) in SCALE else "horizon")
            crow.append(dict(Protocol=pr, Model=m, Scale=smode, Calibration=cb, Nominal=1 - CFG["ALPHA"], Pointwise_coverage=r["point_cov"],
                             Trajectory_coverage=r["traj_cov"], Mean_width_z=r["width"], Median_fold_change_halfwidth=r["fold"]))
CONF = pd.DataFrame(crow)
save_table(CONF.round(4), "T4_conformal", caption=f"Table 4. Conformal bands at nominal {100 * (1 - CFG['ALPHA']):.0f}% on the test set")

alphas = np.round(np.arange(0.05, 0.55, 0.05), 2)
curve = []
for pr in PROTOCOLS:
    for m in [REF, "GRU", LNODE, PROPOSED, SYMB]:
        for a in alphas:
            r, _ = conformal_eval(m, pr, a)
            row = dict(Protocol=pr, Model=m, alpha=a, nominal=1 - a, traj_cov=r["traj_cov"], width=r["width"])
            if m == PROPOSED:
                row["traj_cov_pointwise_calibration"] = conformal_eval(m, pr, a, None, "pointwise")[0]["traj_cov"]
            curve.append(row)
CURVE = pd.DataFrame(curve)
CURVE.to_csv(os.path.join(DIRS["tables"], "T4b_conformal_curve.csv"), index=False)

### Re-splits
A single cal/test split is one draw of the coverage. None of the models saw the cal/test lesions, so I pool them and re-split 500 times, with the same splits for every model.

In [ ]:
def lesion_scores(model, proto, scale_mode=None, calib="trajectory"):
    if scale_mode is None:
        scale_mode = "learned" if (model, first_seed(model)) in SCALE else "horizon"
    sub = LP[(LP.model == model) & (LP.seed == first_seed(model)) & (LP.protocol == proto) & (LP.set.isin(["cal", "test"]))].copy()
    sub["sc"] = score_scale(sub, scale_mode)
    sub["s"] = sub.abs_err / sub.sc
    S = sub.groupby("pidx").s.max()
    Wmean = sub.groupby("pidx").sc.sum()
    npts = sub.groupby("pidx").size()
    pts = sub.groupby("pidx").s.apply(np.array)
    return S, Wmean, npts, pts

N_CAL = len(SPLIT["cal"])
def resplit(model, proto, scale_mode=None, calib="trajectory", B=None, alpha=None):
    B, alpha = B or CFG["N_RESPLIT"], alpha or CFG["ALPHA"]
    S, Wsum, npts, pts = lesion_scores(model, proto, scale_mode)
    ids = np.sort(S.index.values)
    rng = np.random.default_rng(2024)
    cov, wid = [], []
    for _ in range(B):
        perm = rng.permutation(ids)
        cal, te = perm[:N_CAL], perm[N_CAL:]
        Sc = S[cal].values if calib == "trajectory" else np.concatenate(pts[cal].values)
        k = int(math.ceil((len(Sc) + 1) * (1 - alpha)))
        q = float(np.sort(Sc)[min(k, len(Sc)) - 1])
        cov.append(float((S[te].values <= q).mean()))
        wid.append(float(2 * q * Wsum[te].sum() / npts[te].sum()))
    return np.array(cov), np.array(wid)

rs_rows, RS_RAW = [], {}
for pr in PROTOCOLS:
    ref_w = resplit(REF, pr)[1]
    for m in CONF_MODELS:
        variants = [(None, "trajectory")] + ([("horizon", "trajectory"), (None, "pointwise")] if m == PROPOSED else [])
        for sm, cb in variants:
            c, w = resplit(m, pr, sm, cb)
            smode = sm or ("learned" if (m, first_seed(m)) in SCALE else "horizon")
            RS_RAW[(pr, m, smode, cb)] = (c, w)
            ratio = w / ref_w
            rs_rows.append(dict(Protocol=pr, Model=m, Scale=smode, Calibration=cb, Mean_traj_coverage=c.mean(), SD_coverage=c.std(),
                                P05_coverage=np.percentile(c, 5), Mean_width_z=w.mean(), Width_ratio_vs_LOCF=ratio.mean(),
                                Width_ratio_CI95=f"[{np.percentile(ratio, 2.5):.3f}, {np.percentile(ratio, 97.5):.3f}]"))
RSPLIT = pd.DataFrame(rs_rows)
save_table(RSPLIT.round(4), "T4c_conformal_resplits", caption=f"Table 4c. Conformal coverage and width over {CFG['N_RESPLIT']} random calibration/test re-splits (nominal {1 - CFG['ALPHA']:.2f})")

## Early progression (P2)
A test lesion counts as progressing if at some later scan its diameter is ≥ 1.2× the running nadir and at least 5 mm above it (a one-lesion version of RECIST 1.1). Score = biggest forecast rise of z above its running minimum.

In [ ]:
def pd_label(p, obs_k):
    d = p["diam"]
    for j in range(obs_k, len(d)):
        nadir = d[:j].min()
        if d[j] >= 1.2 * nadir and d[j] - nadir >= 5.0:
            return 1
    return 0

items_p2 = ITEMS[("P2", "test")]
y_pd = np.array([pd_label(PATIENTS[i], len(o)) for i, o, _ in items_p2])
log(f"Progression prevalence in the P2 test set: {y_pd.mean():.3f} ({y_pd.sum()}/{len(y_pd)})")

def pd_score(model):
    sc = []
    for (i, o, tg), zp in zip(items_p2, PRED[(model, first_seed(model))][("P2", "test")]):
        zz = np.concatenate([PATIENTS[i]["z"][o], zp])
        sc.append(max(zz[j] - zz[:j].min() for j in range(len(o), len(zz))))
    return np.array(sc)

rng = np.random.default_rng(0)
boot_idx = [rng.integers(0, len(y_pd), len(y_pd)) for _ in range(CFG["N_BOOT"])]
auc_rows, ROC = [], {}
valid_boot = [ix for ix in boot_idx if 0 < y_pd[ix].sum() < len(ix)]
sc_ref = pd_score(REF)
for m in MAIN_ORDER:
    sc = pd_score(m)
    auc = roc_auc_score(y_pd, sc)
    b = np.array([roc_auc_score(y_pd[ix], sc[ix]) for ix in valid_boot])
    bref = np.array([roc_auc_score(y_pd[ix], sc_ref[ix]) for ix in valid_boot])
    diff = b - bref
    fpr, tpr, _ = roc_curve(y_pd, sc)
    ROC[m] = (fpr, tpr, auc)
    auc_rows.append(dict(Model=m, AUC=auc, CI95=f"[{np.percentile(b, 2.5):.3f}, {np.percentile(b, 97.5):.3f}]",
                         Diff_vs_LOCF=auc - roc_auc_score(y_pd, sc_ref),
                         Diff_CI95=f"[{np.percentile(diff, 2.5):.3f}, {np.percentile(diff, 97.5):.3f}]",
                         p_boot_two_sided=fmt_p(min(1.0, 2 * min((diff <= 0).mean(), (diff >= 0).mean()))) if m != REF else "-"))
AUC = pd.DataFrame(auc_rows)
save_table(AUC.round(4), "T6_progression_auc", caption=f"Table 6. Early progression prediction from three scans (n = {len(y_pd)}, prevalence {y_pd.mean():.2f})")

## Results by trial

In [ ]:
ps = []
for pr in PROTOCOLS:
    for m in MAIN_ORDER:
        sub = sub_of(m, pr)
        for s in STUDIES:
            ss = sub[sub.study == s]
            if len(ss):
                ps.append(dict(Protocol=pr, Model=m, Study=f"Study {s}", n=ss.pidx.nunique(), MAE_z=per_patient_mae(ss).mean()))
PS = pd.DataFrame(ps)
save_table(PS.pivot_table(index=["Protocol", "Model"], columns="Study", values="MAE_z", sort=False).reset_index().round(4),
           "T7_per_trial_mae", caption="Table 7. Test MAE (log relative volume) by trial")

## Figures

In [ ]:
MODEL_COL = {PROPOSED: PAL[0], REF: PAL[1], SYMB: PAL[2], LNODE: PAL[3], "GRU": PAL[4], "Gen. Bertalanffy": PAL[6], "Stein-EB": PAL[5], MECH: PAL[7]}
colr = lambda m: MODEL_COL.get(m, GRAY)

In [ ]:
# Fig 2: training curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for m in [PROPOSED, LNODE, MECH, "GRU"]:
    for s in SEEDS:
        h = HIST[(m, s)]
        sm_ = h.mae.rolling(25, min_periods=1).mean()
        ax[0].plot(h.epoch, sm_, color=colr(m), alpha=1.0 if s == SEEDS[0] else 0.35, lw=2 if s == SEEDS[0] else 1,
                   label=m if s == SEEDS[0] else None)
ax[0].set(xlabel="Epoch", ylabel="Training MAE (z), 25-epoch mean",
          title="(a) Forecast-trained models (all seeds; new random cut-offs every epoch)")
ax[0].legend(fontsize=8)
for m in ["Stein-EB", UDEMAP_NAME]:
    h = HIST[(m, SEEDS[0])]
    ax[1].plot(h.epoch, h.mae, color=colr(m), label=m)
ax[1].set(xlabel="Iteration", ylabel="Training fit MAE (z)", title="(b) Population curve-fitting models")
ax[1].legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "F2_training_curves")

In [ ]:
# Fig 3: MAE with bootstrap CIs
fig, ax = plt.subplots(1, 2, figsize=(14, 5.4), sharey=True)
for a, pr in zip(ax, PROTOCOLS):
    vals, cis = [], []
    for m in MAIN_ORDER:
        x = per_patient_mae(sub_of(m, pr)).values
        vals.append(x.mean()); cis.append(boot_ci(x))
    vals, cis = np.array(vals), np.array(cis)
    y = np.arange(len(MAIN_ORDER))
    cols = [PAL[0] if m == PROPOSED else PAL[2] if m == SYMB else PAL[1] if m == REF else GRAY for m in MAIN_ORDER]
    a.barh(y, vals, color=cols, height=0.7, edgecolor="white", linewidth=2)
    a.errorbar(vals, y, xerr=[vals - cis[:, 0], cis[:, 1] - vals], fmt="none", ecolor=INK2, elinewidth=1.2, capsize=3)
    for yy, v, c in zip(y, vals, cis[:, 1]):
        a.text(c + 0.015, yy, f"{v:.3f}", va="center", fontsize=8, color=INK2)
    a.set_yticks(y, MAIN_ORDER)
    a.invert_yaxis()
    a.set(xlabel="Mean absolute error, log relative volume", title=PROTO_DESC[pr])
    a.grid(axis="y", visible=False)
fig.tight_layout()
save_fig(fig, "F3_forecast_accuracy")

In [ ]:
# Fig 4: error vs horizon
LINE_MODELS = [PROPOSED, SYMB, REF, LNODE, "GRU", "Gen. Bertalanffy"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)
edges = np.array([0, 60, 120, 180, 270, 400, 800]) / TU
for a, pr in zip(ax, PROTOCOLS):
    for m in LINE_MODELS:
        sub = sub_of(m, pr).copy()
        sub["bin"] = pd.cut(sub.horizon, edges)
        g = sub.groupby("bin", observed=True).abs_err.agg(["mean", "count"])
        g = g[g["count"] >= 10]
        a.plot(np.array([iv.mid for iv in g.index]) * TU, g["mean"], marker="o", ms=5, color=colr(m), label=m)
    a.set(xlabel="Forecast horizon (days after last observed scan)", ylabel="MAE (z)", title=PROTO_DESC[pr])
ax[0].legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "F4_error_vs_horizon")

In [ ]:
# Fig 5: a few P2 forecasts with their bands
_, te_u = conformal_eval(PROPOSED, "P2", CFG["ALPHA"])
_, te_l = conformal_eval(REF, "P2", CFG["ALPHA"])
cand = te_u.groupby("pidx").size()
cand = cand[cand >= 5].index.values
show = np.random.default_rng(3).choice(cand, size=min(6, len(cand)), replace=False)
fig, axs = plt.subplots(2, 3, figsize=(15, 7.5))
k = CFG["N_EARLY_P2"]
for a, i in zip(axs.ravel(), show):
    p = PATIENTS[i]
    u = te_u[te_u.pidx == i].sort_values("t")
    l = te_l[te_l.pidx == i].sort_values("t")
    a.fill_between(l.t * TU, l.lo, l.hi, color=PAL[1], alpha=0.10, lw=0, label="Persistence 90% band")
    a.fill_between(u.t * TU, u.lo, u.hi, color=PAL[0], alpha=0.22, lw=0, label="UDE 90% band")
    a.plot(u.t * TU, u.z_pred, color=PAL[0], marker="o", ms=4, label="UDE forecast")
    g_ = sub_of("Gen. Bertalanffy", "P2")
    g_ = g_[g_.pidx == i].sort_values("t")
    a.plot(g_.t * TU, g_.z_pred, color=PAL[6], ls="--", marker="s", ms=4, label="Gen. Bertalanffy")
    a.plot(p["t"][:k] * TU, p["z"][:k], "o", color=INK, ms=8, label="Observed (input)")
    a.plot(p["t"][k:] * TU, p["z"][k:], "o", mfc="white", mec=INK, ms=8, mew=1.5, label="Held-out truth")
    a.axvline(p["t"][k - 1] * TU, color=INK2, ls=":", lw=1)
    a.set(title=f"Study {p['study']}, lesion {p['pid'][:6]}", xlabel="Days from baseline", ylabel="z")
    a.set_ylim(max(ZMIN, min(p["z"].min(), u.lo.min()) - 0.3), min(ZMAX, max(p["z"].max(), u.hi.max()) + 0.3))
axs[0, 0].legend(fontsize=7, loc="best")
fig.tight_layout()
save_fig(fig, "F5_example_forecasts_P2")

In [ ]:
# Fig 6: network vs recovered law
zz = np.linspace(np.percentile(Zs, 1), np.percentile(Zs, 99), 120)
tt = np.linspace(np.percentile(Ts, 1), np.percentile(Ts, 99), 120)
ZZ, TT = np.meshgrid(zz, tt)
with torch.no_grad():
    G_nn = ude0.g(torch.tensor(ZZ.ravel(), dtype=torch.float32), torch.tensor(TT.ravel(), dtype=torch.float32)).numpy().reshape(ZZ.shape)
G_sym = (library_np(ZZ.ravel(), TT.ravel()) @ (coef_sym * active_sym)).reshape(ZZ.shape)
vmin, vmax = np.percentile(G_nn, 1), np.percentile(G_nn, 99)
fig, ax = plt.subplots(1, 4, figsize=(19, 4.3))
for a, G, ttl in [(ax[0], G_nn, "(a) Learned network g(z, t)"), (ax[1], G_sym, "(b) Recovered symbolic law")]:
    im = a.pcolormesh(TT * TU, ZZ, G, cmap="Blues", vmin=vmin, vmax=vmax, shading="auto")
    fig.colorbar(im, ax=a, fraction=0.046)
    a.set(xlabel="Days from baseline", ylabel="z", title=ttl)
    a.grid(False)
dmax = max(np.abs(G_nn - G_sym).max(), 1e-6)
im = ax[2].pcolormesh(TT * TU, ZZ, G_nn - G_sym, cmap="RdBu_r", vmin=-dmax, vmax=dmax, shading="auto")
fig.colorbar(im, ax=ax[2], fraction=0.046)
ax[2].set(xlabel="Days from baseline", ylabel="z", title="(c) Network minus symbolic law")
ax[2].grid(False)
ax[3].plot(par.n_terms, par.R2, marker="o", color=GRAY)
ax[3].plot([chosen.n_terms], [chosen.R2], marker="o", ms=11, color=PAL[2], label=f"Selected ({int(chosen.n_terms)} terms)")
ax[3].set(xlabel="Active library terms", ylabel="R$^2$ against network", title="(d) Sparsity-fidelity Pareto front")
ax[3].legend()
fig.tight_layout()
save_fig(fig, "F6_symbolic_recovery")

In [ ]:
# Fig 7: conformal calibration
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
for a, pr, tag in zip(ax[:2], PROTOCOLS, "ab"):
    a.plot([0.45, 0.95], [0.45, 0.95], color=INK2, ls="--", lw=1, label="Ideal")
    for m in [PROPOSED, SYMB, REF, LNODE, "GRU"]:
        c = CURVE[(CURVE.Protocol == pr) & (CURVE.Model == m)]
        a.plot(c.nominal, c.traj_cov, marker="o", ms=5, color=colr(m), label=m)
    c = CURVE[(CURVE.Protocol == pr) & (CURVE.Model == PROPOSED)]
    a.plot(c.nominal, c.traj_cov_pointwise_calibration, marker="x", ms=6, ls=":", color=PAL[0], label="UDE, pointwise calibration")
    a.set(xlabel="Nominal coverage", ylabel="Empirical whole-trajectory coverage", title=f"({tag}) {PROTO_DESC[pr]}")
    a.legend(fontsize=7)
cw = RSPLIT[(RSPLIT.Calibration == "trajectory") & (RSPLIT.Protocol == "P1")].drop_duplicates("Model").set_index("Model").loc[CONF_MODELS]
y = np.arange(len(CONF_MODELS))
ax[2].barh(y, cw.Mean_width_z, color=[PAL[0] if m == PROPOSED else PAL[2] if m == SYMB else PAL[1] if m == REF else GRAY for m in CONF_MODELS],
           height=0.7, edgecolor="white", linewidth=2)
for yy, (w_, c_) in enumerate(zip(cw.Mean_width_z, cw.Mean_traj_coverage)):
    ax[2].text(w_ + 0.03, yy, f"{w_:.2f}  (coverage {c_:.2f})", va="center", fontsize=8, color=INK2)
ax[2].set_yticks(y, CONF_MODELS)
ax[2].invert_yaxis()
ax[2].grid(axis="y", visible=False)
ax[2].set(xlabel="Mean 90% band width (z)", title=f"(c) Width and coverage, mean of {CFG['N_RESPLIT']} re-splits, P1")
fig.tight_layout()
save_fig(fig, "F7_conformal")

In [ ]:
# Fig 8: ROC
fig, ax = plt.subplots(figsize=(5.8, 5.2))
ax.plot([0, 1], [0, 1], color=INK2, ls="--", lw=1)
for m in [PROPOSED, SYMB, REF, LNODE, "GRU", "Gen. Bertalanffy"]:
    fpr, tpr, auc = ROC[m]
    ax.plot(fpr, tpr, color=colr(m), label=f"{m} (AUC {auc:.3f})")
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="Early progression prediction (P2)")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
save_fig(fig, "F8_progression_roc")

In [ ]:
# Fig 9: per-trial heatmap (P2)
Hm = PS[PS.Protocol == "P2"].pivot(index="Model", columns="Study", values="MAE_z").loc[MAIN_ORDER]
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(Hm.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(Hm.shape[1]), Hm.columns)
ax.set_yticks(range(Hm.shape[0]), Hm.index)
ax.grid(False)
mid = np.nanmean(Hm.values)
for r_ in range(Hm.shape[0]):
    for c_ in range(Hm.shape[1]):
        v = Hm.values[r_, c_]
        ax.text(c_, r_, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if v > 1.2 * mid else INK)
fig.colorbar(im, ax=ax, fraction=0.046, label="MAE (z)")
ax.set_title("Test MAE by trial, protocol P2")
fig.tight_layout()
save_fig(fig, "F9_per_trial_heatmap")

In [ ]:
# Fig 11: coverage / width over re-splits
RS_MODELS = [REF, "GRU", LNODE, PROPOSED, SYMB]
fig, ax = plt.subplots(1, 4, figsize=(19, 4.3))
for c_, pr in enumerate(PROTOCOLS):
    covs = [RS_RAW[(pr, m, "learned" if (m, first_seed(m)) in SCALE else "horizon", "trajectory")][0] for m in RS_MODELS]
    covs.append(RS_RAW[(pr, PROPOSED, "learned", "pointwise")][0])
    wids = [RS_RAW[(pr, m, "learned" if (m, first_seed(m)) in SCALE else "horizon", "trajectory")][1] for m in RS_MODELS]
    labels = RS_MODELS + ["UDE, pointwise calib."]
    cols = [colr(m) for m in RS_MODELS] + [GRAY]
    a = ax[2 * c_]
    bp = a.boxplot(covs, patch_artist=True, widths=0.6, medianprops=dict(color=INK), showfliers=False)
    for patch, col in zip(bp["boxes"], cols):
        patch.set_facecolor(col); patch.set_alpha(0.6); patch.set_edgecolor(INK2)
    a.axhline(1 - CFG["ALPHA"], color=INK2, ls="--", lw=1)
    a.set_xticks(range(1, len(labels) + 1), labels, rotation=35, ha="right", fontsize=8)
    a.set(ylabel="Whole-trajectory coverage", title=f"({'ac'[c_]}) Coverage, {pr}")
    a = ax[2 * c_ + 1]
    bp = a.boxplot(wids, patch_artist=True, widths=0.6, medianprops=dict(color=INK), showfliers=False)
    for patch, col in zip(bp["boxes"], cols[:-1]):
        patch.set_facecolor(col); patch.set_alpha(0.6); patch.set_edgecolor(INK2)
    a.set_xticks(range(1, len(RS_MODELS) + 1), RS_MODELS, rotation=35, ha="right", fontsize=8)
    a.set(ylabel="Mean 90% band width (z)", title=f"({'bd'[c_]}) Band width, {pr}")
fig.tight_layout()
save_fig(fig, "F11_conformal_resplits")

In [ ]:
# Fig 10: rho, kappa by trial
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, col, lab in [(ax[0], "rho", "Net growth rate rho (per 100 days)"), (ax[1], "kappa", "Treatment-effect strength kappa")]:
    data = [PARAMS_TE[PARAMS_TE.study == s][col].values for s in STUDIES]
    bp = a.boxplot(data, patch_artist=True, widths=0.6, medianprops=dict(color=INK))
    for patch, s in zip(bp["boxes"], STUDIES):
        patch.set_facecolor(STUDY_COL[s]); patch.set_alpha(0.55); patch.set_edgecolor(INK2)
    a.set_xticks(range(1, len(STUDIES) + 1), [f"Study {s}" for s in STUDIES])
    a.set(ylabel=lab, title=f"Symbolic UDE: {col} inferred from the first three scans")
fig.tight_layout()
save_fig(fig, "F10_patient_parameters")

## Summary file

In [ ]:
TIMINGS["total_s"] = time.time() - T_START
def row_of(pr, m):
    return RES[pr].set_index("Model").loc[m]
summary = dict(
    run_id=RUN_ID, data_source=DATA_SOURCE, data_md5=DATA_MD5, n_lesions=NP, split={k: int(len(v)) for k, v in SPLIT.items()},
    seeds=SEEDS, config=CFG, environment=ENV,
    accuracy={pr: {m: dict(MAE_z=float(row_of(pr, m).MAE_z), MAE_z_sd=None if np.isnan(row_of(pr, m).MAE_z_sd) else float(row_of(pr, m).MAE_z_sd))
                   for m in MAIN_ORDER} for pr in PROTOCOLS},
    noninferiority=NI.to_dict("records"),
    discovered_law=dict(stlsq=law_stlsq, refined=law_refined, r2=float(chosen.R2), n_terms=int(chosen.n_terms)),
    conformal=CONF.to_dict("records"),
    conformal_resplits=RSPLIT.to_dict("records"),
    departure_from_persistence=DEV.to_dict("records"),
    progression=dict(prevalence=float(y_pd.mean()), n=int(len(y_pd)), auc={r.Model: float(r.AUC) for r in AUC.itertuples()}),
    timings_s={k: round(v, 1) for k, v in TIMINGS.items()},
)
with open(os.path.join(OUT, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
save_table(pd.DataFrame([dict(Stage=k, Seconds=round(v, 1)) for k, v in TIMINGS.items()]), "T9_runtime", caption="Table 9. Computational cost on this machine")

files = sorted(os.path.relpath(os.path.join(r, f), OUT) for r, _, fs in os.walk(OUT) for f in fs)
with open(os.path.join(OUT, "MANIFEST.txt"), "w") as f:
    f.write("\n".join(files))
display(Markdown(f"### Finished in {TIMINGS['total_s'] / 60:.1f} minutes\nAll {len(files)} output files are saved in `{OUT}`"))
print("Discovered law:", law_refined)
display(RES["P1"][["Model", "MAE_z", "MAE_z_CI95"]].round(4))
display(RES["P2"][["Model", "MAE_z", "MAE_z_CI95"]].round(4))